# Thesis overview figure (adapted from BEELINE Fig. 1, Pratapa et al.)

This recreates the three-panel structure of BEELINE's Figure 1 (data → methods → evaluation) for the calibration thesis. It uses generic drawn icons instead of the original's software logos.

**Everything you may want to change is in `TEXT` and `PALETTE` at the top of the next cell.** Several labels are my reading of the thesis plan and should be checked:
- Model box: "$f_\theta(x)=\hat p$ (edge probability)". This fits Tracks 2–3. Track 1 predicts a label probability, not an edge probability.
- "Generalisation bounds (Tracks 1–2 only)": the PAC-Bayes part is not implemented yet and still needs scoping with the supervisor.
- The Baselines list (temperature, Platt, isotonic, Beta; focal loss) and the metric list reflect the planned comparison.

Outputs: `thesis_overview.pdf` (vector, for LaTeX), `.svg`, and a 300-dpi `.png`.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle, Circle

# ------------------------------------------------------------------ config
PALETTE = {
    "data": ("#EEF0FA", "#3949AB"),     # (fill, accent)
    "train": ("#FFF4E8", "#E07B00"),
    "eval": ("#E8F5F3", "#00897B"),
    "text": "#1F2430", "muted": "#5A6072", "card": "#FFFFFF",
}
TEXT = {
    "panel_data": "Data",
    "panel_train": "Training & calibration",
    "panel_eval": "Evaluation",
    "tracks": [
        ("Track 1 · Synthetic 1-D", "Known posterior $\\eta(x)$\nLegendre basis models"),
        ("Track 2 · SERGIO", "Simulated scRNA-seq, known GRN\nper-(cell, pair) features, cell split"),
        ("Track 3 · DREAM5", "Real benchmark\nempirical case study"),
    ],
    "iid_label": "i.i.d. by construction",
    "model": ("Probabilistic classifier", "$f_\\theta(x) = \\hat{p}$  (edge probability)"),
    "e2e": ("End-to-end calibration",
            "$\\min_\\theta\\; \\mathcal{L}_{\\mathrm{base}} + \\lambda\\,\\widehat{\\mathrm{CE}}$",
            "base: CE, Brier\n$\\widehat{\\mathrm{CE}}$: Soft-ECE, MMCE, $\\mathrm{ECE}_{\\mathrm{KDE}}$"),
    "baselines": ("Baselines", "Post-hoc: temperature,\nPlatt, isotonic, Beta\nTraining: focal loss"),
    "sweep": ("Hyperparameter sweep", "$\\lambda$ grid, capacity $m$"),
    "preds": ("Predicted probabilities", "$\\hat{p}$ per test sample"),
    "evals": [
        ("Calibration", "ECE, reliability diagrams"),
        ("Calibration–refinement", "risk = calibration + refinement"),
        ("Discrimination", "AUROC, AUPR, Early Precision"),
        ("Robustness", "RobustnessGap under shift"),
        ("Generalisation bounds", "PAC-Bayes, Rademacher\n(Tracks 1–2 only)"),
    ],
}
FONT = "DejaVu Sans"

# ------------------------------------------------------------------ helpers
def box(ax, x, y, w, h, fc, ec, lw=1.0, r=1.2, z=1, ls="-"):
    p = FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={r}",
                       fc=fc, ec=ec, lw=lw, zorder=z, ls=ls)
    ax.add_patch(p); return p

def arrow(ax, p0, p1, color, lw=1.6, style="-|>", ms=12, conn="arc3", z=5, ls="-"):
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle=style, mutation_scale=ms, color=color,
                                 lw=lw, connectionstyle=conn, zorder=z, linestyle=ls,
                                 shrinkA=0, shrinkB=0))

def inset(ax, x, y, w, h):
    # axes inset in data coordinates of the canvas
    a = ax.inset_axes([x, y, w, h], transform=ax.transData)
    a.set_xticks([]); a.set_yticks([])
    for s in a.spines.values(): s.set_visible(False)
    a.patch.set_alpha(0)
    return a

def title_sub(ax, x, y, title, sub, accent, tsize=9.5, ssize=7.6, ha="left"):
    ax.text(x, y, title, fontsize=tsize, fontweight="bold", color=PALETTE["text"], ha=ha, va="top", zorder=6)
    ax.text(x, y - 2.1, sub, fontsize=ssize, color=PALETTE["muted"], ha=ha, va="top",
            linespacing=1.35, zorder=6)

# ---- mini icons
rng = np.random.default_rng(3)
def icon_curve(a, c):
    x = np.linspace(-1, 1, 200); eta = 1/(1+np.exp(-4*np.sin(2.2*x)))
    xs = rng.uniform(-1, 1, 40); ys = (rng.uniform(size=40) < 1/(1+np.exp(-4*np.sin(2.2*xs)))).astype(float)
    a.scatter(xs, ys*1.0, s=6, color=c, alpha=.5, lw=0)
    a.plot(x, eta, color=c, lw=1.6); a.set_xlim(-1.1, 1.1); a.set_ylim(-.15, 1.15)
    a.axhline(0, color="#B0B4C0", lw=.6); 

def icon_network(a, c):
    n = 7; th = np.linspace(0, 2*np.pi, n, endpoint=False) + .3
    xy = np.c_[np.cos(th), np.sin(th)]
    E = [(0,1),(1,2),(2,3),(0,4),(4,5),(5,6),(6,2),(3,5),(1,6)]
    for i, j in E:
        a.annotate("", xy[j], xy[i], arrowprops=dict(arrowstyle="-|>", color="#8A90A6", lw=.7,
                   shrinkA=4, shrinkB=4, mutation_scale=6))
    a.scatter(xy[:,0], xy[:,1], s=26, color=c, zorder=3, ec="white", lw=.8)
    a.set_xlim(-1.35, 1.35); a.set_ylim(-1.35, 1.35); a.set_aspect("equal")

def icon_matrix(a, c):
    M = rng.gamma(1.2, 1, (7, 10)); M[2, 3:6] += 3; M[5, 6:] += 2.5
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list("m", ["#FFFFFF", c])
    a.imshow(M, cmap=cmap, aspect="auto", interpolation="nearest")
    for s in a.spines.values(): s.set_visible(True); s.set_color("#B0B4C0"); s.set_linewidth(.6)

def icon_reliability(a, c):
    edges = np.linspace(0, 1, 7)[:-1]; w = 1/6; acc = np.clip(edges + w/2 - .12*edges, 0, 1)
    a.bar(edges, acc, w, align="edge", color=c, alpha=.85, ec="white", lw=.6)
    a.bar(edges, (edges+w/2)-acc, w, bottom=acc, align="edge", color="#E53935", alpha=.25, ec="#E53935", lw=.5, hatch="///")
    a.plot([0,1],[0,1], ls="--", color="#6B7080", lw=.8); a.set_xlim(0,1); a.set_ylim(0,1)
    a.spines["left"].set_visible(True); a.spines["bottom"].set_visible(True)
    for s in ("left","bottom"): a.spines[s].set_color("#9095A5"); a.spines[s].set_linewidth(.6)

def icon_decomp(a, c):
    for v, left, col, lab in [(.35, 0, c, "cal"), (.65, .35, "#E07B00", "ref")]:
        a.barh(0, v, left=left, color=col, alpha=.85, height=.6, ec="white", lw=.8)
        a.text(left + v/2, 0, lab, ha="center", va="center", fontsize=6.3, color="white", fontweight="bold")
    a.set_xlim(0, 1); a.set_ylim(-.6, .6)

def icon_roc(a, c):
    t = np.linspace(0, 1, 100)
    a.plot(t, t**.35, color=c, lw=1.6); a.plot(t, t**.6, color=c, lw=1.1, alpha=.45)
    a.plot([0,1],[0,1], ls="--", color="#6B7080", lw=.8); a.set_xlim(0,1); a.set_ylim(0,1.02)
    for s in ("left","bottom"): a.spines[s].set_visible(True); a.spines[s].set_color("#9095A5"); a.spines[s].set_linewidth(.6)

def icon_shift(a, c):
    x = np.linspace(-4, 5, 200)
    a.fill_between(x, np.exp(-x**2/2), color=c, alpha=.35, lw=0)
    a.plot(x, np.exp(-x**2/2), color=c, lw=1.2)
    a.plot(x, .8*np.exp(-(x-1.8)**2/2.6), color="#E07B00", lw=1.2, ls="--")
    a.set_ylim(0, 1.1)

def icon_bound(a, c):
    n = np.linspace(1, 10, 100)
    a.plot(n, .15 + .9/np.sqrt(n), color="#E07B00", lw=1.3, ls="--")
    a.plot(n, .15 + .35/np.sqrt(n) + .02*np.sin(3*n), color=c, lw=1.5)
    a.set_ylim(.1, 1.1)
    for s in ("left","bottom"): a.spines[s].set_visible(True); a.spines[s].set_color("#9095A5"); a.spines[s].set_linewidth(.6)

def icon_search(a, c):
    a.add_patch(Circle((.42, .58), .26, fc="white", ec=c, lw=2.2))
    a.plot([.61, .85], [.39, .13], color=c, lw=3, solid_capstyle="round")
    a.set_xlim(0, 1); a.set_ylim(0, 1); a.set_aspect("equal")

def icon_hist(a, c):
    v = np.clip(rng.beta(4, 1.6, 600), 0, 1)
    a.hist(v, bins=12, color=c, alpha=.85, ec="white", lw=.6); a.set_xlim(0, 1)
    a.spines["bottom"].set_visible(True); a.spines["bottom"].set_color("#9095A5"); a.spines["bottom"].set_linewidth(.6)

# ------------------------------------------------------------------ figure
def draw(save_stem=None):
    plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "dejavusans",
                         "pdf.fonttype": 42, "svg.fonttype": "none"})
    W, H = 100, 56
    fig = plt.figure(figsize=(12, 12*H/W))
    ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, W); ax.set_ylim(0, H); ax.axis("off")
    dF, dA = PALETTE["data"]; tF, tA = PALETTE["train"]; eF, eA = PALETTE["eval"]

    # panel geometry
    PY, PH = 2, 52
    P1, P2, P3 = (1.5, 27), (34.5, 31), (72.5, 26)   # (x, w)
    for (x, w), fc, ac, name in [(P1, dF, dA, TEXT["panel_data"]), (P2, tF, tA, TEXT["panel_train"]),
                                 (P3, eF, eA, TEXT["panel_eval"])]:
        box(ax, x, PY, w, PH, fc, "none", r=2)
        ax.text(x + w/2, PY + PH - 2.2, name.upper(), ha="center", va="top", fontsize=10.5,
                fontweight="bold", color=ac)

    # ---- data cards
    cx, cw, ch = P1[0] + 1.6, P1[1] - 3.2, 12.0
    ys = [36.0, 22.0, 5.2]
    icons = [icon_curve, icon_network, icon_matrix]
    for (t, s), y, ic in zip(TEXT["tracks"], ys, icons):
        box(ax, cx, y, cw, ch, PALETTE["card"], "#D5D9EA", lw=.8, r=1)
        ax.add_patch(Rectangle((cx, y + .9), .55, ch - 1.8, fc=dA, ec="none", zorder=3))
        title_sub(ax, cx + 1.6, y + ch - 1.3, t, s, dA, tsize=8.8, ssize=7.0)
        ic(inset(ax, cx + cw - 7.2, y + 1.2, 6.4, 5.4), dA)
    # i.i.d. bracket (tracks 1-2)
    box(ax, cx - .8, ys[1] - .9, cw + 1.6, ys[0] + ch - ys[1] + 1.8, "none", dA, lw=1.0, r=1.4, ls=(0, (4, 3)), z=2)
    ax.text(cx + cw/2, ys[1] - 1.6, TEXT["iid_label"], ha="center", va="top", fontsize=7.4,
            color=dA, style="italic", bbox=dict(fc=dF, ec="none", pad=1.2), zorder=4)

    # ---- training panel
    bx, bw = P2[0] + 1.6, P2[1] - 3.2
    # model
    my, mh = 40.2, 7.2
    box(ax, bx, my, bw, mh, PALETTE["card"], "#EBD6C0", lw=.8, r=1)
    title_sub(ax, bx + bw/2, my + mh - 1.2, TEXT["model"][0], TEXT["model"][1], tA, ha="center", ssize=8.2)
    # two branches
    by, bh, gap = 20.6, 15.2, 1.4
    lw_ = bw*0.56; rw_ = bw - lw_ - gap
    box(ax, bx, by, lw_, bh, PALETTE["card"], tA, lw=1.4, r=1)
    t, eq, sub = TEXT["e2e"]
    ax.text(bx + lw_/2, by + bh - 1.2, t, ha="center", va="top", fontsize=9.2, fontweight="bold", color=tA)
    ax.text(bx + lw_/2, by + bh - 5.3, eq, ha="center", va="center", fontsize=9.6, color=PALETTE["text"])
    ax.text(bx + lw_/2, by + 1.3, sub, ha="center", va="bottom", fontsize=7.2, color=PALETTE["muted"], linespacing=1.4)
    rx = bx + lw_ + gap
    box(ax, rx, by, rw_, bh, PALETTE["card"], "#EBD6C0", lw=.8, r=1, ls=(0, (3, 2)))
    ax.text(rx + rw_/2, by + bh - 1.2, TEXT["baselines"][0], ha="center", va="top", fontsize=9.2, fontweight="bold", color=PALETTE["text"])
    ax.text(rx + rw_/2, by + bh/2 - 1.2, TEXT["baselines"][1], ha="center", va="center", fontsize=7.2, color=PALETTE["muted"], linespacing=1.45)
    # sweep + predictions
    sy, sh = 4.2, 12.0
    sw = bw*0.46
    box(ax, bx, sy, sw, sh, PALETTE["card"], "#EBD6C0", lw=.8, r=1)
    title_sub(ax, bx + sw/2, sy + sh - 1.3, TEXT["sweep"][0], TEXT["sweep"][1], tA, ha="center", tsize=7.9, ssize=7.2)
    icon_search(inset(ax, bx + sw/2 - 2.6, sy + .8, 5.2, 5.2), tA)
    px, pw = bx + sw + gap + 2.2, bw - sw - gap - 2.2
    box(ax, px, sy, pw, sh, PALETTE["card"], "#EBD6C0", lw=.8, r=1)
    title_sub(ax, px + pw/2, sy + sh - 1.3, TEXT["preds"][0], TEXT["preds"][1], tA, ha="center", tsize=7.9, ssize=7.2)
    icon_hist(inset(ax, px + 2.2, sy + 1.0, pw - 4.4, 4.6), tA)
    # internal arrows
    arrow(ax, (bx + bw/2, my), (bx + bw/2, by + bh + .1), tA, lw=1.3, ms=10)
    arrow(ax, (bx + sw/2, by), (bx + sw/2, sy + sh + .1), tA, lw=1.3, ms=10)
    arrow(ax, (bx + sw, sy + sh/2), (px - .1, sy + sh/2), tA, lw=1.3, ms=10)

    # ---- data -> training arrows (merge)
    jx = P2[0] - 2.4
    for y, h in zip(ys, [ch]*3):
        ax.plot([cx + cw, jx], [y + h/2, y + h/2], color="#7A8095", lw=1.4, zorder=4, solid_capstyle="round")
    ax.plot([jx, jx], [ys[2] + ch/2, my + mh/2], color="#7A8095", lw=1.4, zorder=4)
    arrow(ax, (jx, my + mh/2), (bx - .1, my + mh/2), "#7A8095", lw=1.4)

    # ---- evaluation cards
    ex, ew = P3[0] + 1.6, P3[1] - 3.2
    n = len(TEXT["evals"]); top, bottom = PY + PH - 5.0, PY + 1.4
    eh = (top - bottom - (n - 1)*1.2) / n
    eicons = [icon_reliability, icon_decomp, icon_roc, icon_shift, icon_bound]
    centers = []
    for i, ((t, s), ic) in enumerate(zip(TEXT["evals"], eicons)):
        y = top - (i + 1)*eh - i*1.2
        box(ax, ex, y, ew, eh, PALETTE["card"], "#C9E3DF", lw=.8, r=1)
        ax.add_patch(Rectangle((ex, y + .8), .55, eh - 1.6, fc=eA, ec="none", zorder=3))
        title_sub(ax, ex + 1.6, y + eh - 1.1, t, s, eA, tsize=8.6, ssize=6.9)
        a = inset(ax, ex + ew - 6.4, y + 1.0, 5.4, eh - 2.0)
        ic(a, eA)
        centers.append(y + eh/2)
    # fan-out from predictions to eval cards
    fx0, fx1 = P2[0] + P2[1] - 1.6, P3[0] - 1.2
    ax.plot([px + pw, fx0 + .1], [sy + sh/2, sy + sh/2], color="#7A8095", lw=1.4, zorder=4)
    ax.plot([fx0 + .1, fx0 + 1.4], [sy + sh/2, sy + sh/2], color="#7A8095", lw=1.4, zorder=4)
    vx = fx0 + 1.4
    ax.plot([vx, vx], [min(centers + [sy + sh/2]), max(centers)], color="#7A8095", lw=1.4, zorder=4)
    for c in centers:
        arrow(ax, (vx, c), (ex - .1, c), "#7A8095", lw=1.4, ms=10)

    if save_stem:
        for ext in ("pdf", "png", "svg"):
            fig.savefig(f"{save_stem}.{ext}", dpi=300, bbox_inches="tight", pad_inches=0.05)
    return fig

In [ ]:
fig = draw(save_stem="thesis_overview")
plt.show()

### LaTeX include (suggested)
```latex
\begin{figure}[t]
  \centering
  \includegraphics[width=\textwidth]{figures/thesis_overview.pdf}
  \caption{Overview of the evaluation framework. Three data tracks (left) feed a common
  training pipeline in which end-to-end calibration is compared against post-hoc and
  focal-loss baselines (middle). Predicted probabilities are evaluated for calibration,
  calibration--refinement, discrimination, robustness, and, in the i.i.d.\ tracks,
  generalisation bounds (right). Layout adapted from Pratapa et al.\ (BEELINE), Fig.~1.}
  \label{fig:overview}
\end{figure}
```